# Notebook 03: Advanced Tuning + Experiment Tracking

Goal: teach repeatable experimentation and hyperparameter tuning.


In [ ]:
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

from src.config import PROCESSED_DIR
from src.data import load_raw_data, basic_cleaning, make_text_feature, split_data
from src.features import build_text_tabular_preprocessor


In [ ]:
df = load_raw_data()
df = basic_cleaning(df)
df['text'] = make_text_feature(df)
X_train, X_valid, y_train, y_valid = split_data(df)
print(X_train.shape, X_valid.shape)


## Baseline advanced model score (CV)


In [ ]:
base_pipeline = Pipeline([
    ('prep', build_text_tabular_preprocessor(text_col='text')),
    ('model', RandomForestClassifier(
        n_estimators=300,
        max_depth=16,
        min_samples_leaf=2,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1
    ))
])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
base_cv_f1 = cross_val_score(base_pipeline, X_train, y_train, scoring='f1', cv=cv, n_jobs=1)
print('Base CV F1 mean:', base_cv_f1.mean())
print('Base CV F1 std :', base_cv_f1.std())


## Optuna tuning loop


In [ ]:
import optuna

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 700, step=100),
        'max_depth': trial.suggest_int('max_depth', 8, 30),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
    }

    pipe = Pipeline([
        ('prep', build_text_tabular_preprocessor(text_col='text')),
        ('model', RandomForestClassifier(
            **params,
            class_weight='balanced_subsample',
            random_state=42,
            n_jobs=-1
        ))
    ])

    score = cross_val_score(pipe, X_train, y_train, scoring='f1', cv=cv, n_jobs=1).mean()
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)
print('Best score:', study.best_value)
print('Best params:', study.best_trial.params)


In [ ]:
# Track experiments in a simple JSONL log
exp_dir = PROCESSED_DIR / 'experiments'
exp_dir.mkdir(parents=True, exist_ok=True)
log_file = exp_dir / 'tuning_runs.jsonl'

record = {
    'timestamp': datetime.now().isoformat(timespec='seconds'),
    'dataset_rows': int(len(df)),
    'target': 'Recommended IND',
    'model': 'RandomForestClassifier + TFIDF + tabular',
    'best_f1_cv': float(study.best_value),
    'best_params': study.best_trial.params,
}

with open(log_file, 'a', encoding='utf-8') as f:
    f.write(json.dumps(record) + '\n')

print('Logged experiment to:', log_file)
record


## Next experiments for students

- Compare RandomForest vs LightGBM.
- Add threshold tuning for F1 optimization.
- Create a train/validation drift check report.
- Track all runs and choose final model by robust metrics.
